In [15]:
# %% make repo root importable (so `from src...` works)
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

root on path: /Users/admin/Desktop/carbon-portfolio-project-v2


In [16]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import pandas as pd
from src.db import connect
from src.data_download import batch_download_strict_min_points
from src.price_pull import prep_symbols, load_prices
from src.fx_pull import currencies_in_universe, fetch_fx, load_fx


In [18]:
con = connect(str(ROOT / 'data' / 'carbon.db'))

In [19]:

# %% one-time: apply new prices/fx_rates schema (prices is empty -> safe to drop)
con.executescript("DROP TABLE IF EXISTS prices;")
con.executescript((ROOT / 'sql' / 'schema.sql').read_text())


In [20]:
print(pd.read_sql("PRAGMA table_info(prices);", con)[['name']].to_string(index=False))
print(pd.read_sql("PRAGMA table_info(fx_rates);", con)[['name']].to_string(index=False))

      name
company_id
      date
      open
      high
       low
     close
    volume
  currency
    source
        name
        date
    currency
rate_per_eur
      source


In [21]:

# %% Section 1 — symbol prep (inspect, confirm before pulling)
ok, unmapped = prep_symbols(con)

tickers total     : 8,179
mapped to yahoo   : 6,406
unmapped exchange : 1,773
currency mix      :
currency
EUR    2647
GBp    1268
PLN     695
SEK     660
RON     294
NOK     243
CHF     204
USD     146
DKK     134
HUF      62
CZK      21
AUD      13
CAD       7
HKD       5
ZAc       3
ISK       2
ILA       2
Unmapped Exchange:
 exchange
Delisted                        724
Bulgarian Stock Exchange        242
Spotlight Stock Market          131
Nordic Growth Market (NGM)       93
Mercado Alternativo Bursatil     91
Cyprus Stock Exchange            85
Zagreb Stock Exchange            65
Malta Stock Exchange             53
OTC Bulletin Board               37
Norvegian OTC                    36


In [27]:
raw = pd.read_sql("SELECT ticker FROM master_company_list WHERE ticker LIKE '%.'", con)
print(f"trailing-dot tickers: {len(raw)}")
print(raw['ticker'].head(20).to_string(index=False))

trailing-dot tickers: 14
AO.
AT.
BA.
BP.
JD.
NG.
QQ.
RE.
RM.
RR.
SN.
UU.
VP.
YU.


In [24]:
ok['universe'].value_counts()

universe
EU     6122
ETS     284
Name: count, dtype: int64

In [25]:
# %% SMOKE TEST — 50 symbols only, confirm the path before the full backfill
symbol_to_id  = dict(zip(ok['yahoo_symbol'], ok['company_id']))
symbol_to_ccy = dict(zip(ok['yahoo_symbol'], ok['currency']))

smoke = ok['yahoo_symbol'].head(50).tolist()

smoke_df, smoke_short, smoke_never = batch_download_strict_min_points(
    tickers    = smoke,
    start      = "2013-01-01", end = "2025-06-30",
    auto_adjust= False,
    batch_size = 50, pause = 20, min_count = 6,
    output_csv = None, no_data_csv = None,
)
load_prices(con, smoke_df, symbol_to_id, symbol_to_ccy)

Total tickers to process: 50


$UU-.L: possibly delisted; no timezone found
$BP-.L: possibly delisted; no timezone found

2 Failed downloads:
['UU-.L', 'BP-.L']: possibly delisted; no timezone found



Tickers with < 6 non-NaN Close values: ['BP-.L', 'UU-.L']

Final: 48 tickers with >= 6 data, 2 with too little data, 0 never returned any data.
prices upserted: 155,520 rows, 48 companies


In [ ]:
# %% Section 2 — price backfill (auto_adjust=False -> raw prices)
filtered_df, too_short, never_seen = batch_download_strict_min_points(
    tickers    = ok['yahoo_symbol'].tolist(),
    start      = "2013-01-01", end = "2025-06-30",
    auto_adjust= False,
    batch_size = 50, pause = 20, min_count = 6,
    output_csv = "data/raw/prices_backfill.csv",
    no_data_csv= "data/raw/prices_missing.csv",
)



In [ ]:
# %% Section 2b — load prices
symbol_to_id  = dict(zip(ok['yahoo_symbol'], ok['company_id']))
symbol_to_ccy = dict(zip(ok['yahoo_symbol'], ok['currency']))
load_prices(con, filtered_df, symbol_to_id, symbol_to_ccy)


In [ ]:

# %% Section 3 — FX pull
ccys = currencies_in_universe(con)
print("currencies to pull:", ccys)
fx_df = fetch_fx(ccys)
load_fx(con, fx_df)


In [ ]:

# %% Section 4 — emissions as-of   (next)
# %% Section 5 — coverage log       (next)